In [1]:
!pip install scipy --break-system-packages -q

import re
import html
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from scipy import stats

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "resep_masakan"

model = SentenceTransformer(MODEL_NAME)
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)
print("Jumlah dokumen ter-index:", collection.count())


def add_query_prefix(text: str) -> str:
    return f"query: {text}"


def add_passage_prefix(text: str) -> str:
    return f"passage: {text}"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2992.24it/s]


Jumlah dokumen ter-index: 5000


In [ ]:
UNITS = [
    "sendok makan",
    "sendok teh",
    "sdm",
    "sdt",
    "gram",
    "gr",
    "kg",
    "ons",
    "ml",
    "mili",
    "liter",
    "ltr",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ikat",
    "ruas",
    "cm",
    "potong",
    "ptg",
    "biji",
    "bj",
    "ekor",
    "ekr",
    "genggam",
    "bungkus",
    "bks",
    "sachet",
    "sct",
    "papan",
    "buku",
    "helai",
    "hlai",
    "gelas",
    "gls",
    "cup",
    "mangkuk",
    "piring",
    "sisir",
    "kotak",
    "ktk",
    "kaleng",
    "klg",
    "botol",
    "btl",
]
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)
QTY_PATTERN = (
    r"\d+\s*-\s*\d+"
    r"|\d+\s+\d+\s*/\s*\d+"
    r"|\d+\s*/\s*\d+"
    r"|\d+[.,]\d+"
    r"|\d+"
    r"|sesuai selera"
    r"|secukupnya"
    r"|sejumput"
    r"|beberapa"
)

INGREDIENT_REGEX = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?:(?P<unit>{UNIT_PATTERN})\b)?\.?\s*(?P<name>.+?)\s*$",
    re.IGNORECASE,
)


def clean_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"\(.*?\)", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def parse_ingredient(raw: str) -> dict:
    raw = html.unescape(raw).strip()
    raw = re.sub(r"^[\-\*•]+\s*", "", raw)
    main, note = (raw.split(",", 1) + [""])[:2] if "," in raw else (raw, "")
    main = main.strip()
    m = INGREDIENT_REGEX.match(main)
    name = clean_name(m.group("name")) if (m and m.group("name")) else clean_name(main)
    return {"raw": raw, "name": name}


def parse_user_ingredients(text: str) -> list:
    items = [t.strip() for t in text.split(",") if t.strip()]
    parsed = [parse_ingredient(item) for item in items]
    return [p["name"] for p in parsed if p["name"]]


def ingredient_matches(u: str, r: str) -> bool:
    u, r = u.lower().strip(), r.lower().strip()
    if not u or not r:
        return False
    return u in r or r in u


def coverage_stats(user_ingredients: list, recipe_ingredients: list):
    if not recipe_ingredients:
        return 0.0, 0.0, [], []
    matched_recipe_items, matched_user_items = [], set()
    for r_name in recipe_ingredients:
        hit = False
        for u_name in user_ingredients:
            if ingredient_matches(u_name, r_name):
                hit = True
                matched_user_items.add(u_name)
        if hit:
            matched_recipe_items.append(r_name)
    missing_items = [r for r in recipe_ingredients if r not in matched_recipe_items]
    coverage_recipe = len(matched_recipe_items) / len(recipe_ingredients)
    coverage_user = (
        len(matched_user_items) / len(user_ingredients) if user_ingredients else 0.0
    )
    return coverage_recipe, coverage_user, matched_recipe_items, missing_items


def harmonic_efficiency(cr, cu):
    return (2 * cr * cu) / (cr + cu) if (cr + cu) else 0.0


def fbeta_efficiency(cr, cu, beta=2.0):
    denom = (beta**2 * cr) + cu
    return (1 + beta**2) * (cr * cu) / denom if denom else 0.0

In [4]:
def retrieve_candidates(user_text: str, top_k: int = 30) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    candidates = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        candidates.append(
            {
                "id": results["ids"][0][i],
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "sbert_score": 1 - results["distances"][0][i],
            }
        )
    return candidates


_ALL_DOCS_CACHE = None


def get_all_documents():
    global _ALL_DOCS_CACHE
    if _ALL_DOCS_CACHE is None:
        raw = collection.get(include=["metadatas", "documents", "embeddings"])
        _ALL_DOCS_CACHE = list(
            zip(raw["ids"], raw["metadatas"], raw["documents"], raw["embeddings"])
        )
    return _ALL_DOCS_CACHE


def keyword_candidates(
    user_ingredients: list, exclude_ids: set, max_candidates: int = 20
) -> list:
    scored = []
    for doc_id, meta, document, embedding in get_all_documents():
        if doc_id in exclude_ids:
            continue
        names = json.loads(meta["ingredient_names"])
        matched = sum(
            1 for u in user_ingredients if any(ingredient_matches(u, n) for n in names)
        )
        if matched > 0:
            scored.append((matched, doc_id, meta, document, embedding))
    scored.sort(key=lambda x: -x[0])
    return scored[:max_candidates]


def retrieve_candidates_hybrid(
    user_text: str,
    user_ingredients: list,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    sbert_candidates = retrieve_candidates(user_text, top_k=top_k_sbert)
    existing_ids = {c["id"] for c in sbert_candidates}
    kw_extra = keyword_candidates(
        user_ingredients, exclude_ids=existing_ids, max_candidates=max_keyword_extra
    )
    for matched, doc_id, meta, document, embedding in kw_extra:
        sbert_score = float(np.dot(query_embedding, np.array(embedding)))
        sbert_candidates.append(
            {
                "id": doc_id,
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "sbert_score": sbert_score,
            }
        )
    return sbert_candidates

## Skenarionya

In [5]:
def scenario_sbert_only(user_text: str, top_n: int = 5, top_k_sbert: int = 30) -> list:
    candidates = retrieve_candidates(user_text, top_k=top_k_sbert)
    return sorted(candidates, key=lambda c: c["sbert_score"], reverse=True)[:top_n]


def scenario_reranked(
    user_text: str,
    top_n: int = 5,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
    formula: str = "harmonic",
    w_sbert: float = 0.3,
    w_efficiency: float = 0.7,
    beta: float = 2.0,
) -> list:
    """formula: 'harmonic' pakai harmonic_efficiency, 'fbeta' pakai fbeta_efficiency.
    Sesuaikan w_sbert/w_efficiency/beta dengan formula final yang kamu pilih dari notebook 03."""
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates_hybrid(
        user_text, user_ingredients, top_k_sbert, max_keyword_extra
    )

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        cr, cu, matched_items, missing_items = coverage_stats(
            user_ingredients, c["ingredient_names"]
        )
        eff = (
            fbeta_efficiency(cr, cu, beta=beta)
            if formula == "fbeta"
            else harmonic_efficiency(cr, cu)
        )
        c["coverage_recipe"] = cr
        c["matched_items"] = matched_items
        c["missing_items"] = missing_items
        c["final_score"] = w_sbert * float(s_norm) + w_efficiency * eff

    return sorted(candidates, key=lambda c: c["final_score"], reverse=True)[:top_n]


def scenario_keyword_only(user_text: str, top_n: int = 5) -> list:
    user_ingredients = parse_user_ingredients(user_text)
    scored = []
    for doc_id, meta, document, embedding in get_all_documents():
        names = json.loads(meta["ingredient_names"])
        cr, cu, matched_items, missing_items = coverage_stats(user_ingredients, names)
        if cu > 0:
            scored.append(
                {
                    "id": doc_id,
                    "title": meta["title"],
                    "ingredient_names": names,
                    "coverage_recipe": cr,
                    "coverage_user": cu,
                    "matched_items": matched_items,
                    "missing_items": missing_items,
                }
            )
    scored.sort(key=lambda c: (c["coverage_user"], c["coverage_recipe"]), reverse=True)
    return scored[:top_n]

## Metrik: IAR & Average Missing Ingredients Count

In [6]:
def compute_query_metrics(user_text: str, ranked_results: list, k: int) -> dict:
    """IAR = rata-rata coverage_recipe (bahan resep yg sudah dimiliki user / total bahan resep) pada top-k.
    Missing = rata-rata jumlah bahan yang masih perlu ditambah pada top-k."""
    user_ingredients = parse_user_ingredients(user_text)
    top_k_results = ranked_results[:k]
    if not top_k_results:
        return {"iar": 0.0, "missing": 0.0}

    iars, missing_counts = [], []
    for r in top_k_results:
        if "coverage_recipe" in r and "missing_items" in r:
            cr, missing = r["coverage_recipe"], len(r["missing_items"])
        else:
            cr, cu, matched_items, missing_items = coverage_stats(
                user_ingredients, r["ingredient_names"]
            )
            missing = len(missing_items)
        iars.append(cr)
        missing_counts.append(missing)

    return {"iar": float(np.mean(iars)), "missing": float(np.mean(missing_counts))}

## Query Uji

In [11]:
test_queries = [
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih",
    "200gr daging sapi, kecap manis, bawang bombay",
    "tahu, tempe, cabai rawit",
    "udang, tepung terigu, telur",
    "wortel, kentang, buncis",
    "tempe, kacang panjang, cabai merah",
    "2 butir telur",
    "nasi sisa kemarin, 1 butir telur",
    "1 papan tempe, kecap manis",
    "dada ayam fillet dikit",
    "indomie goreng, sawi hijau, cabe rawit",
    "roti tawar, keju cheddar",
    "1 buah tahu, 1 buah tomat, daun bawang",
    "500 gr daging sapi, 1 buah bawang bombay, kecap inggris",
    "1 ikat bayam, jagung manis, bawang merah",
    "kangkung, terasi, cabai rawit, bawang merah",
    "ikan nila, bumbu kuning, kemangi",
    "sosis sapi, nasi putih, bawang bombay, saus sambal",
    "200 gr udang, brokoli, bawang putih, saus tiram",
    "ikan lele, jeruk nipis, garam, ketumbar",
    "500gr ayam, santan kental, daun salam, serai, lengkuas, bawang merah, bawang putih",
    "daging sapi, kelapa sangrai, cabai merah besar, bawang merah, bawang putih, jahe, lengkuas",
    "nasi putih, telur, sosis, bakso, kecap manis, saus tomat, sayur kol, daun bawang",
    "dada ayam, tepung bumbu, telur, bawang putih, saus asam manis, nanas, paprika",
    "tahu, tempe, kacang panjang, tauge, bumbu kacang, kerupuk",
    "tepung terigu, mentega, gula pasir, telur, vanili",
    "pisang kepok, tepung beras, gula merah, minyak goreng",
    "bubuk matcha murni, susu cair, es batu",
    "250 gr jamur enoki, gochujang, minyak wijen",
    "frozen kentang goreng, sosis sapi, bumbu tabur jagung bakar",
]
print(f"Jumlah query uji: {len(test_queries)}")

Jumlah query uji: 30


## Evaluasi Otomatis untuk Semua Query

In [12]:
K_VALUES = [3, 5]
rows = []

for q in test_queries:
    res_sbert = scenario_sbert_only(q, top_n=max(K_VALUES))
    res_reranked = scenario_reranked(q, top_n=max(K_VALUES))
    res_keyword = scenario_keyword_only(q, top_n=max(K_VALUES))

    row = {"query": q}
    for k in K_VALUES:
        row[f"sbert_iar@{k}"] = compute_query_metrics(q, res_sbert, k)["iar"]
        row[f"sbert_missing@{k}"] = compute_query_metrics(q, res_sbert, k)["missing"]
        row[f"reranked_iar@{k}"] = compute_query_metrics(q, res_reranked, k)["iar"]
        row[f"reranked_missing@{k}"] = compute_query_metrics(q, res_reranked, k)[
            "missing"
        ]
        row[f"keyword_iar@{k}"] = compute_query_metrics(q, res_keyword, k)["iar"]
        row[f"keyword_missing@{k}"] = compute_query_metrics(q, res_keyword, k)[
            "missing"
        ]
    rows.append(row)

df_eval = pd.DataFrame(rows)
df_eval.to_csv("evaluasi_otomatis_per_query.csv", index=False)
df_eval

,query,sbert_iar@3,sbert_missing@3,reranked_iar@3,reranked_missing@3,keyword_iar@3,keyword_missing@3,sbert_iar@5,sbert_missing@5,reranked_iar@5,reranked_missing@5,keyword_iar@5,keyword_missing@5
0,"2 butir telur, 100gr dada ayam, 2 siung bawang...",0.209524,5.000000,0.301587,4.666667,0.452381,3.666667,0.249524,4.8,0.323810,4.6,0.421429,4.2
1,"200gr daging sapi, kecap manis, bawang bombay",0.259615,9.333333,0.388889,5.000000,0.420635,4.333333,0.217674,10.0,0.357143,5.2,0.385714,5.0
2,"tahu, tempe, cabai rawit",0.122782,12.666667,0.233333,8.000000,0.206090,11.666667,0.135574,9.8,0.237143,7.4,0.358948,9.8
3,"udang, tepung terigu, telur",0.283730,5.666667,0.467857,3.666667,0.509524,3.000000,0.356905,4.6,0.447381,4.0,0.455714,3.8
4,"wortel, kentang, buncis",0.186508,9.333333,0.205556,8.000000,0.325000,6.333333,0.181602,9.4,0.193030,8.6,0.299545,7.2
5,"tempe, kacang panjang, cabai merah",0.243386,6.333333,0.295238,6.333333,0.271429,8.333333,0.236032,6.6,0.300952,5.6,0.296190,6.6
6,2 butir telur,0.245370,5.333333,0.466667,1.666667,0.583333,1.000000,0.272222,4.8,0.396667,2.0,0.550000,1.0
7,"nasi sisa kemarin, 1 butir telur",0.166667,6.666667,0.500000,1.333333,0.333333,4.666667,0.131515,8.8,0.394444,3.4,0.288889,5.6
8,"1 papan tempe, kecap manis",0.203704,7.333333,0.322222,5.000000,0.333333,4.000000,0.192991,8.2,0.277778,6.0,0.326667,4.6
9,dada ayam fillet dikit,0.051852,13.333333,0.157407,8.333333,0.250000,3.000000,0.031111,13.6,0.164444,7.4,0.230000,3.4


## Rangkuman Agregat (Rata-rata Semua Query)

In [13]:
summary_rows = []
for scenario in ["sbert", "reranked", "keyword"]:
    for k in K_VALUES:
        summary_rows.append(
            {
                "scenario": scenario,
                "k": k,
                "mean_IAR": df_eval[f"{scenario}_iar@{k}"].mean(),
                "mean_missing_ingredients": df_eval[f"{scenario}_missing@{k}"].mean(),
            }
        )

df_summary = pd.DataFrame(summary_rows)
df_summary

,scenario,k,mean_IAR,mean_missing_ingredients
0,sbert,3,0.210654,7.877778
1,sbert,5,0.199127,8.220000
2,reranked,3,0.351304,5.344444
3,reranked,5,0.321379,5.733333
4,keyword,3,0.338850,6.900000
5,keyword,5,0.320755,7.700000


## Uji Signifikansi (Wilcoxon Signed-Rank Test)

Membandingkan `reranked` vs `sbert_only` (metode yang diusulkan vs baseline) secara berpasangan per query. `H0`: tidak ada perbedaan median antara kedua skenario. Kalau `p-value < 0.05`, peningkatan (atau penurunan) dianggap signifikan secara statistik, bukan kebetulan.

In [14]:
for k in K_VALUES:
    a = df_eval[f"sbert_iar@{k}"].values
    b = df_eval[f"reranked_iar@{k}"].values
    diff = b - a
    if np.all(diff == 0):
        print(
            f"IAR@{k}: semua selisih = 0, Wilcoxon tidak bisa dihitung (tidak ada variasi)."
        )
        continue
    stat, p = stats.wilcoxon(a, b)
    direction = "naik" if b.mean() > a.mean() else "turun"
    print(
        f"IAR@{k}: SBERT-only mean={a.mean():.3f}, Reranked mean={b.mean():.3f} ({direction})"
    )
    print(
        f"         Wilcoxon statistic={stat:.3f}, p-value={p:.4f} -> "
        f"{'SIGNIFIKAN' if p < 0.05 else 'tidak signifikan'} (alpha=0.05)"
    )
    print()

IAR@3: SBERT-only mean=0.211, Reranked mean=0.351 (naik)
         Wilcoxon statistic=0.000, p-value=0.0000 -> SIGNIFIKAN (alpha=0.05)

IAR@5: SBERT-only mean=0.199, Reranked mean=0.321 (naik)
         Wilcoxon statistic=0.000, p-value=0.0000 -> SIGNIFIKAN (alpha=0.05)

